# 11 · Explainability (Grad-CAM)

**Purpose.** Show where each ensemble member looks. A trustworthy model looks at the lesion, not the background.

| | |
|---|---|
| **Reads** | `best_models.json`, `test_predictions.csv` |
| **Writes** | `reports/figures/gradcam_<class>.png`, `reports/results/gradcam_review.csv` |
| **Run time (Colab T4)** | ~5 min |
| **Next notebook** | 12 · Source-Shortcut Test |

Run the cells from top to bottom. All shared logic lives in `src/dermalens/`; this
notebook only calls it and explains each decision.

## 0. Setup

The same cell starts every notebook. It (1) connects Google Drive on Colab,
(2) installs missing libraries on Colab, (3) makes `src/` importable,
(4) finds every project folder and (5) loads `configs/baseline.yaml`.

In [ ]:
import os
import sys
from pathlib import Path

# 1) On Colab, connect Google Drive so the project folder is visible.
try:
    from google.colab import drive                      # this import only works on Colab
    drive.mount("/content/drive")
    os.environ.setdefault("DERMALENS_PROJECT_DIR", "/content/drive/MyDrive/DermaLens_AI")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False                                    # local run: set DERMALENS_PROJECT_DIR yourself

# 2) Where the data lives (PROJECT_DIR) and where the code lives (CODE_DIR, normally the same folder).
PROJECT_DIR = Path(os.environ.get("DERMALENS_PROJECT_DIR", Path.cwd().parent))
CODE_DIR = Path(os.environ.get("DERMALENS_CODE_DIR", PROJECT_DIR))

# 3) Colab starts empty every session: install the few libraries it does not ship with.
if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(CODE_DIR / "requirements.txt")], check=True)

# 4) Make `import dermalens` work.
sys.path.insert(0, str(CODE_DIR / "src"))

from dermalens.config import load_config
from dermalens.paths import ProjectPaths
from dermalens.utils import set_seed

# 5) Folders and settings.
paths = ProjectPaths(PROJECT_DIR).ensure()
CONFIG = load_config(os.environ.get("DERMALENS_CONFIG", CODE_DIR / "configs" / "baseline.yaml"))
set_seed(CONFIG["seed"])
print(paths.describe())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dermalens.data import make_loader, resolve_images
from dermalens.utils import get_device, load_json, save_json

info = load_json(paths.splits / "class_info.json")       # classes + weights frozen in notebook 05
CLASSES = info["classes"]                                  # 6 targets + "Other"
WEIGHTS = [info["class_weights"][name] for name in CLASSES]
train = pd.read_csv(paths.splits / "train.csv")
val = pd.read_csv(paths.splits / "val.csv")
IMAGE_ROOT, PATH_COL = resolve_images(train, paths)        # cache if present, else raw files
DEVICE = get_device()
IMAGE_SIZE = CONFIG["data"]["image_size"]
BATCH, WORKERS = CONFIG["train"]["batch_size"], CONFIG["train"]["num_workers"]
USE_META, COLOR = CONFIG["data"]["use_metadata"], CONFIG["data"]["color_constancy"]


def loader(rows, train_mode=False, size=IMAGE_SIZE, use_meta=USE_META, classes=None, label_col="label",
           seed=0):
    """One DataLoader with the project settings (augmentation + metadata dropout only when training)."""
    return make_loader(rows, IMAGE_ROOT, classes or CLASSES, size, train=train_mode, batch_size=BATCH,
                       num_workers=WORKERS, path_col=PATH_COL, label_col=label_col, use_metadata=use_meta,
                       meta_dropout=CONFIG["data"]["meta_dropout"], color_constancy=COLOR, seed=seed)


print(f"train {len(train)} | val {len(val)} | classes {len(CLASSES)} | device {DEVICE} | "
      f"images from {IMAGE_ROOT} ({PATH_COL})")

from PIL import Image
from dermalens import metadata as meta
from dermalens.data import build_transforms, center_square, load_rgb
from dermalens.explain import GradCAM, overlay_heatmap
from dermalens.model import gradcam_target_layer
from dermalens.train import model_from_checkpoint

predictions = pd.read_csv(paths.results / "test_predictions.csv")
member = load_json(paths.results / "best_models.json")["members"][0]
model, facts = model_from_checkpoint(paths.root / member["checkpoint"], DEVICE)
cam = GradCAM(model, gradcam_target_layer(model))
to_tensor = build_transforms(facts["image_size"], train=False, color_constancy=COLOR)
print("Explaining", member["model"], "- works for CNNs, ViT (token maps) and Swin")

Per class: the 3 most confident correct predictions and up to 3 wrong ones. Red = evidence for the predicted class.

In [ ]:
review_rows = []
for label in CLASSES:
    in_class = predictions[predictions["true_label"] == label]
    chosen = pd.concat([in_class[in_class["correct"]].nlargest(3, "confidence"),
                        in_class[~in_class["correct"]].nlargest(3, "confidence")])
    if chosen.empty:
        continue
    fig, axes = plt.subplots(2, len(chosen), figsize=(2.8 * len(chosen), 5.6), squeeze=False)
    for column, (_, item) in enumerate(chosen.iterrows()):
        photo = load_rgb(IMAGE_ROOT / item[PATH_COL])
        vector = meta.encode_row(item) if facts["use_metadata"] else None
        heatmap, _, _ = cam(to_tensor(photo), meta_vector=vector)
        shown = center_square(photo).resize((facts["image_size"], facts["image_size"]))
        axes[0, column].imshow(shown)
        axes[1, column].imshow(overlay_heatmap(shown, heatmap))
        verdict = "correct" if item["correct"] else f"WRONG: {item['predicted']}"
        axes[0, column].set_title(f"{verdict}\n{item['confidence']:.0%}", fontsize=8,
                                  color="green" if item["correct"] else "red")
        for ax in axes[:, column]:
            ax.axis("off")
        review_rows.append({"image_id": item["image_id"], "true_label": label, "predicted": item["predicted"],
                            "correct": item["correct"], "attention_on_lesion (yes/no/partly)": "", "notes": ""})
    fig.suptitle(f"True class: {label}")
    plt.tight_layout()
    plt.savefig(paths.figures / f"gradcam_{label.replace(' ', '_').replace('/', '').lower()}.png", dpi=130)
    plt.show()
cam.remove()
pd.DataFrame(review_rows).to_csv(paths.results / "gradcam_review.csv", index=False)
print(f"{len(review_rows)} images to review by a teammate -> gradcam_review.csv")